# 📦 QLoRA: LoRA + Quantization = Fine-Tuning for Everyone!

In the previous notebook, we learned how LoRA makes fine-tuning cheaper by only training small adapter matrices. QLoRA takes this even further by **compressing** the frozen model weights.

## 🎯 What You'll Learn
- What quantization is (and why it's like using fewer crayons)
- How QLoRA combines quantization with LoRA
- Why QLoRA lets you fine-tune huge models on consumer GPUs
- The key innovations that make QLoRA work
- Real-world usage patterns

## 📋 Prerequisites
- Completed the LoRA notebook
- Basic Python knowledge

---
## 1️⃣ The Problem: Even LoRA Needs Memory for the Frozen Model

LoRA solved the training cost problem - we only train tiny adapter matrices. But there's still a catch:

```
  LoRA Memory Usage:
  
  ┌─────────────────────────────────────────────────────┐
  │  Frozen model weights: Still in GPU memory!         │
  │                                                     │
  │  7B model in FP16 = ~14 GB                          │
  │  LoRA adapters    = ~0.1 GB                         │
  │  Activations      = ~2-4 GB                         │
  │  ──────────────────────────                         │
  │  Total: ~16-18 GB                                   │
  │                                                     │
  │  A consumer GPU (RTX 3090) has 24 GB                │
  │  → 7B model barely fits!                            │
  │  → 13B model? NO WAY! ❌                            │
  └─────────────────────────────────────────────────────┘
```

**QLoRA's solution:** Compress the frozen weights to take up much less memory!

```
  QLoRA Memory Usage:
  
  ┌─────────────────────────────────────────────────────┐
  │  Frozen model (4-bit quantized) = ~3.5 GB  (was 14!)│
  │  LoRA adapters (FP16)           = ~0.1 GB           │
  │  Activations                    = ~2-4 GB           │
  │  ──────────────────────────                         │
  │  Total: ~6-8 GB                                     │
  │                                                     │
  │  A consumer GPU (RTX 3090) has 24 GB                │
  │  → 7B model fits EASILY! ✅                         │
  │  → 13B model? Yes! ✅                               │
  │  → 33B model? Barely, but YES! ✅                   │
  └─────────────────────────────────────────────────────┘
```

---
## 2️⃣ What is Quantization?

**Quantization** is a way to compress numbers so they take up less memory. The trade-off: slightly less precise, but much smaller.

### 🖍️ The Crayon Analogy

```
  Imagine you're coloring a picture:
  
  FP32 (32-bit float):  A box of 16 MILLION crayons 🖍️
  ├── Every possible shade exists
  ├── "Sunset orange #FF6B35" vs "sunset orange #FF6B36"
  ├── Extremely precise
  └── But... who needs 16 million shades?
  
  FP16 (16-bit float):  A box of 65,536 crayons 🖍️
  ├── Very precise, covers all common shades
  ├── Half the storage of FP32
  └── Most AI uses this
  
  INT8 (8-bit integer): A box of 256 crayons 🖍️
  ├── Good coverage, some shades missing
  ├── 1/4 the storage of FP32
  └── Looks almost the same
  
  INT4 (4-bit integer): A box of 16 crayons 🖍️
  ├── Basic colors only
  ├── 1/8 the storage of FP32!
  └── Surprisingly, still looks decent! ← QLoRA uses this!
```

### 📏 What Happens to Numbers During Quantization

```
  Original number (FP16):     0.123456789
  After INT8 quantization:    0.12
  After INT4 quantization:    0.1
  
  We lose precision, but save A LOT of memory:
  
  FP32:  32 bits per number  │████████████████████████████████│
  FP16:  16 bits per number  │████████████████│
  INT8:   8 bits per number  │████████│
  INT4:   4 bits per number  │████│  ← QLoRA!
```

In [ ]:
# 📊 Let's see quantization in action!

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Simulate model weights (normally distributed, centered near 0)
original_weights = np.random.randn(1000) * 0.1

def quantize(values, bits):
    """Simulate quantization: map continuous values to discrete levels.
    
    Think of it as: reducing the number of 'crayons' available.
    """
    n_levels = 2 ** bits  # 4 bits = 16 levels, 8 bits = 256 levels
    
    # Find the range
    min_val = values.min()
    max_val = values.max()
    
    # Map to discrete levels
    step_size = (max_val - min_val) / (n_levels - 1)
    quantized = np.round((values - min_val) / step_size) * step_size + min_val
    
    return quantized, n_levels, step_size

# Quantize at different bit levels
results = {}
for bits in [16, 8, 4, 2]:
    quantized, n_levels, step = quantize(original_weights, bits)
    error = np.mean(np.abs(original_weights - quantized))
    results[bits] = {
        'quantized': quantized,
        'n_levels': n_levels,
        'error': error,
        'memory_ratio': 32 / bits  # Compared to FP32
    }

# Print comparison
print("Quantization Comparison")
print("=" * 65)
print(f"{'Bits':>6} | {'Levels':>8} | {'Avg Error':>12} | {'Memory Savings':>15}")
print("-" * 65)
print(f"{'32':>6} | {'16M+':>8} | {'0 (exact)':>12} | {'1x (baseline)':>15}")
for bits, info in results.items():
    print(f"{bits:>6} | {info['n_levels']:>8,} | {info['error']:>12.6f} | {info['memory_ratio']:>14.0f}x smaller")

print(f"\nQLoRA uses 4-bit quantization: {results[4]['memory_ratio']:.0f}x memory savings!")
print(f"Average error is only {results[4]['error']:.6f} - barely noticeable!")

In [ ]:
# 📊 Visualize what quantization does to the weights

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

configs = [
    (16, 'FP16 (16-bit) - Standard', '#1565C0'),
    (8, 'INT8 (8-bit) - Good compression', '#2E7D32'),
    (4, 'INT4 (4-bit) - QLoRA uses this!', '#E65100'),
    (2, 'INT2 (2-bit) - Too much compression', '#C62828'),
]

for ax, (bits, title, color) in zip(axes.flat, configs):
    quantized = results[bits]['quantized']
    error = results[bits]['error']
    
    # Plot original vs quantized for first 100 weights
    x = np.arange(100)
    ax.plot(x, original_weights[:100], 'b-', alpha=0.3, linewidth=1, label='Original')
    ax.plot(x, quantized[:100], color=color, linewidth=1.5, label=f'Quantized ({bits}-bit)')
    
    ax.set_title(f'{title}\n(Avg error: {error:.6f})', fontweight='bold', fontsize=11)
    ax.set_xlabel('Weight index')
    ax.set_ylabel('Value')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('How Quantization Affects Model Weights\n(Blue = original, colored = quantized)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('quantization_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("Notice how 4-bit (QLoRA) still closely follows the original values!")
print("Only 2-bit shows significant deviation - that's why QLoRA uses 4-bit.")

---
## 3️⃣ QLoRA = Quantization + LoRA

Now let's put it together! QLoRA combines three key innovations:

### 🏗️ The Three Innovations of QLoRA

```
  Innovation 1: 4-bit NormalFloat (NF4) Quantization
  ──────────────────────────────────────────────────
  A special 4-bit format designed specifically for neural network
  weights (which follow a bell curve / normal distribution).
  
  Regular 4-bit: Evenly spaced levels  │ ─ ─ ─ ─ ─ ─ ─ ─ │
  NF4:          Concentrated in center │─ ── ──── ── ─│
                                       (more detail where weights cluster)
  
  Innovation 2: Double Quantization
  ──────────────────────────────────────────────────
  Even the quantization constants (scaling factors) get quantized!
  
  Regular: Use FP32 scaling factors (wastes memory)
  QLoRA:   Quantize the scaling factors too (saves ~0.4 GB per 1B params)
  
  Innovation 3: Paged Optimizers
  ──────────────────────────────────────────────────
  If GPU runs out of memory, automatically spill to CPU RAM.
  
  Like when your desk (GPU) is full:
  → Move some papers to a filing cabinet (CPU)
  → Pull them back when needed
  → Slower, but at least it works!
```

### 🔧 How QLoRA Works Step by Step

```
  Step 1: Load model in 4-bit precision
  ┌─────────────────────────────────────────────────────┐
  │  Regular:  7B params × 16 bits = 14 GB              │
  │  QLoRA:    7B params ×  4 bits = 3.5 GB  (4x less!) │
  └─────────────────────────────────────────────────────┘
                         │
                         v
  Step 2: Add LoRA adapters (in FP16 - full precision!)
  ┌─────────────────────────────────────────────────────┐
  │  LoRA adapters are SMALL, so keeping them in FP16   │
  │  doesn't cost much: ~0.05-0.1 GB                    │
  │  But the precision matters for learning!             │
  └─────────────────────────────────────────────────────┘
                         │
                         v
  Step 3: During training
  ┌─────────────────────────────────────────────────────┐
  │  Forward pass:                                       │
  │    1. Dequantize weights (4-bit → 16-bit) on the fly│
  │    2. Compute: output = x × W_dequantized + x × A × B│
  │                                                      │
  │  Backward pass:                                      │
  │    1. Compute gradients (through dequantized path)   │
  │    2. Update ONLY A and B (LoRA adapters)             │
  │    3. Frozen weights stay in 4-bit (never updated)   │
  └─────────────────────────────────────────────────────┘
```

In [ ]:
# 📊 Visualize QLoRA's memory savings

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ── 1. Memory comparison for different model sizes ──
ax = axes[0]

model_sizes = ['1B', '7B', '13B', '33B', '65B']
params_b = [1, 7, 13, 33, 65]

# Memory in GB for different methods
full_ft_mem = [p * 16 for p in params_b]      # ~16 bytes per param (FP32 + optimizer)
lora_mem = [p * 2 + 2 for p in params_b]       # FP16 model + small LoRA
qlora_mem = [p * 0.5 + 1 for p in params_b]    # 4-bit model + small LoRA

x = np.arange(len(model_sizes))
width = 0.25

bars1 = ax.bar(x - width, full_ft_mem, width, label='Full Fine-Tuning', 
               color='#F44336', edgecolor='black')
bars2 = ax.bar(x, lora_mem, width, label='LoRA (FP16)', 
               color='#FF9800', edgecolor='black')
bars3 = ax.bar(x + width, qlora_mem, width, label='QLoRA (4-bit)', 
               color='#4CAF50', edgecolor='black')

ax.set_xlabel('Model Size', fontsize=12)
ax.set_ylabel('GPU Memory Required (GB)', fontsize=12)
ax.set_title('Memory Requirements by Method', fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(model_sizes)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Add GPU reference lines
ax.axhline(y=24, color='blue', linestyle=':', alpha=0.5)
ax.text(4.5, 25, 'RTX 3090 (24GB)', fontsize=8, color='blue', ha='right')
ax.axhline(y=80, color='purple', linestyle=':', alpha=0.5)
ax.text(4.5, 82, 'A100 (80GB)', fontsize=8, color='purple', ha='right')

# ── 2. What fits on what GPU? ──
ax = axes[1]

gpus = ['RTX 3060\n(12GB)', 'RTX 3090\n(24GB)', 'RTX 4090\n(24GB)', 'A100\n(40GB)', 'A100\n(80GB)']
gpu_mem = [12, 24, 24, 40, 80]

# What model size fits with QLoRA?
# Rule of thumb: model_size_B * 0.5 + 1 GB for QLoRA
max_model_qlora = [(mem - 1) / 0.5 for mem in gpu_mem]  # Max model size in B params

colors = ['#E3F2FD', '#BBDEFB', '#90CAF9', '#64B5F6', '#42A5F5']
bars = ax.barh(gpus, max_model_qlora, color=colors, edgecolor='#1565C0', linewidth=1.5)

for bar, size in zip(bars, max_model_qlora):
    label = f'~{size:.0f}B params'
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            label, ha='left', va='center', fontweight='bold', fontsize=10)

ax.set_xlabel('Maximum Model Size (Billions of Parameters)', fontsize=12)
ax.set_title('Largest Model You Can QLoRA Fine-Tune', fontweight='bold', fontsize=14)
ax.grid(True, alpha=0.3, axis='x')
ax.set_xlim(0, 200)

plt.suptitle('QLoRA Makes Large Model Fine-Tuning Accessible!',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('qlora_memory.png', dpi=150, bbox_inches='tight')
plt.show()

print("With QLoRA, you can fine-tune a 33B model on a single consumer GPU!")
print("Without QLoRA, that would require multiple expensive A100s.")

---
## 4️⃣ NF4: A Smarter Way to Quantize

QLoRA introduced a special quantization format called **NF4 (4-bit NormalFloat)**.

### 🎯 The Key Insight

Neural network weights aren't evenly distributed - they follow a **bell curve** (normal distribution). Most weights are near zero, with few extreme values.

```
  Distribution of weights in a typical neural network:
  
  Frequency
  │
  │            ████
  │          ████████
  │        ████████████
  │      ████████████████
  │    ████████████████████
  │  ████████████████████████
  │████████████████████████████
  └──────────────────────────────→ Weight value
  -0.3    -0.1    0    0.1    0.3
  
  Most weights cluster near ZERO!
```

Regular quantization spaces levels evenly, wasting precision on rare extreme values. NF4 puts more levels where the weights actually are:

```
  Regular INT4:  ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  (evenly spaced)  Equal gaps everywhere
  
  NF4:           ─ ─ ── ── ──── ──── ── ── ─ ─
  (smart spacing)   More detail in the middle where
                    most weights live!
```

In [ ]:
# 📊 Visualize NF4 vs regular quantization

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Simulate weight distribution (normal/bell curve)
weights = np.random.randn(10000) * 0.1

# Regular INT4: 16 evenly spaced levels
regular_levels = np.linspace(weights.min(), weights.max(), 16)

# NF4: 16 levels optimized for normal distribution
# These are the actual NF4 quantile values (approximately)
nf4_quantiles = np.array([
    -1.0, -0.6962, -0.5251, -0.3949, -0.2844, -0.1848, -0.0911, 0.0,
    0.0796, 0.1609, 0.2461, 0.3379, 0.4407, 0.5626, 0.7230, 1.0
]) * 0.3  # Scale to match our weight range

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Regular INT4 ──
ax = axes[0]
ax.hist(weights, bins=50, density=True, alpha=0.6, color='#BBDEFB', edgecolor='#1565C0')
for level in regular_levels:
    ax.axvline(x=level, color='red', alpha=0.7, linewidth=1.5)
ax.set_title('Regular INT4 Quantization\n(Evenly spaced levels)', fontweight='bold', fontsize=13)
ax.set_xlabel('Weight Value')
ax.set_ylabel('Frequency')
ax.text(0.05, 0.95, 'Problem: Levels wasted\non rare extreme values!', 
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round', facecolor='#FFCDD2', edgecolor='red'))

# ── NF4 ──
ax = axes[1]
ax.hist(weights, bins=50, density=True, alpha=0.6, color='#C8E6C9', edgecolor='#2E7D32')
for level in nf4_quantiles:
    ax.axvline(x=level, color='green', alpha=0.7, linewidth=1.5)
ax.set_title('NF4 Quantization (QLoRA)\n(Levels concentrated where weights are)', fontweight='bold', fontsize=13)
ax.set_xlabel('Weight Value')
ax.set_ylabel('Frequency')
ax.text(0.05, 0.95, 'Better: More levels where\nweights actually cluster!', 
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round', facecolor='#C8E6C9', edgecolor='green'))

plt.suptitle('NF4: Smarter Quantization for Neural Network Weights',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('nf4_vs_regular.png', dpi=150, bbox_inches='tight')
plt.show()

print("Red/green lines = available quantization levels (16 levels for 4-bit)")
print("NF4 (right) puts more levels in the dense center where most weights live!")
print("This gives better precision with the same number of bits.")

---
## 5️⃣ QLoRA in Practice

Here's what real-world QLoRA fine-tuning looks like:

```python
# Real QLoRA code (using Hugging Face libraries)
# DON'T run this - it requires a GPU and large model download

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

# Step 1: Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,              # Use 4-bit quantization
    bnb_4bit_quant_type="nf4",      # Use NF4 format (QLoRA's innovation)
    bnb_4bit_compute_dtype="float16", # Compute in FP16 for accuracy
    bnb_4bit_use_double_quant=True,  # Double quantization (saves more memory)
)

# Step 2: Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b",
    quantization_config=quantization_config,
)

# Step 3: Add LoRA adapters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
)
model = get_peft_model(model, lora_config)

# Now fine-tune! The 7B model fits on a single 24GB GPU 🎉
```

### 📊 Summary Comparison

| Feature | Full Fine-Tuning | LoRA | QLoRA |
|---------|:---:|:---:|:---:|
| Quality | Best (100%) | Great (95-99%) | Great (95-98%) |
| Memory for 7B | ~112 GB | ~16 GB | ~6 GB |
| GPU needed for 7B | 3× A100 (80GB) | 1× A100 (40GB) | 1× RTX 3090 (24GB) |
| Trainable params | 100% | ~0.1% | ~0.1% |
| Forgetting risk | High | Low | Low |
| Training speed | Slow | Fast | Fast |
| Cost (cloud) | $$$$ | $$ | $ |

In [ ]:
# 📊 Final summary visualization

import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(12, 7))

# Data for radar/comparison chart
methods = ['Full Fine-Tuning', 'LoRA', 'QLoRA']
metrics = ['Quality', 'Memory\nEfficiency', 'Speed', 'Cost\nEfficiency', 'Ease of\nUse']

# Scores out of 10
scores = {
    'Full Fine-Tuning': [10, 2, 3, 2, 5],
    'LoRA':             [9, 7, 8, 7, 8],
    'QLoRA':            [8.5, 10, 8, 10, 9],
}
colors = ['#F44336', '#FF9800', '#4CAF50']

x = np.arange(len(metrics))
width = 0.25

for i, (method, score) in enumerate(scores.items()):
    bars = ax.bar(x + i * width, score, width, label=method, color=colors[i],
                  edgecolor='black', linewidth=1)
    for bar, s in zip(bars, score):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
                f'{s}', ha='center', fontweight='bold', fontsize=9)

ax.set_ylabel('Score (out of 10)', fontsize=12)
ax.set_title('Fine-Tuning Methods Comparison\n(Higher = Better)', 
             fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 12)
ax.grid(True, alpha=0.3, axis='y')

# Add "winner" annotation
ax.annotate('Best overall\nfor most people!', 
           xy=(4.25, 9), xytext=(4.5, 11),
           fontsize=10, fontweight='bold', color='#2E7D32',
           arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))

plt.tight_layout()
plt.savefig('methods_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("QLoRA is the best choice for most people:")
print("  - Nearly as good as full fine-tuning")
print("  - Runs on consumer GPUs")
print("  - Costs a fraction of the price")
print("  - Easy to use with Hugging Face libraries")

---
## 🎯 Key Takeaways

1. **Quantization** compresses numbers to use fewer bits (like using fewer crayons)

2. **QLoRA = 4-bit quantized frozen model + FP16 LoRA adapters**

3. **NF4** is a special 4-bit format that puts more precision where weights cluster

4. **Memory savings are dramatic**: 7B model goes from ~112 GB (full FT) to ~6 GB (QLoRA)

5. **Quality loss is minimal**: QLoRA achieves 95-98% of full fine-tuning quality

6. **Democratization**: QLoRA made fine-tuning accessible to researchers and hobbyists with consumer GPUs

## 🚀 What's Next?

Now we know HOW to fine-tune models efficiently. But what kind of data should we use? The next notebook covers **Instruction Tuning** - how to teach models to follow instructions (like ChatGPT!).

| Next Notebook | What You'll Learn |
|---------------|------------------|
| [Instruction Tuning](../instruction-tuning/01_instruction_tuning.ipynb) | How ChatGPT learned to follow instructions |

---

[Back to Module README](../README.md)